# Track B Math Walkthrough

This notebook explains the math behind the scratch-built GPT-like decoder used in Track B.

## 1. Query, Key, Value projections

For a hidden state matrix $X \in \mathbb{R}^{T \times d}$, we learn three projections:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

If batch size is $B$, sequence length is $T$, embedding width is $d$, and we use $h$ heads, then after reshaping:

$$Q, K, V \in \mathbb{R}^{B \times h \times T \times d_h}, \quad d_h = d / h$$

## 2. Scaled dot-product attention

Each token compares its query against every key:

$$S = \frac{QK^T}{\sqrt{d_h}}$$

The scaling term prevents the dot products from growing too large as the head dimension increases.

## 3. Causal mask

A decoder must not look into the future. We therefore add a mask that sets all positions above the diagonal to $-\infty$ before the softmax:

$$A = \operatorname{softmax}(S + M)$$

where $M_{ij} = -\infty$ if token $i$ is not allowed to attend to token $j$.

## 4. Weighted value aggregation

Attention weights tell us how much of each value vector to mix in:

$$\text{Head}(Q,K,V) = AV$$

Multiple heads are concatenated and projected back into the model width.

## 5. Decoder block residual flow

Each decoder block follows:

1. LayerNorm
2. Causal self-attention
3. Residual add
4. LayerNorm
5. MLP
6. Residual add

This preserves gradient flow and lets the block learn corrections rather than replacing the full representation.

## 6. Next-token objective

The model predicts the next token at every position. With logits $z_t$ and true token $y_t$, we minimize cross-entropy:

$$\mathcal{L} = - \sum_t \log p(y_t \mid y_{<t})$$

Padding positions are ignored by setting their labels to `-100` in PyTorch.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.model import MiniGPTConfig, MiniGPTLM

config = MiniGPTConfig(vocab_size=32, block_size=8, n_layer=1, n_head=2, n_embd=16, dropout=0.0)
model = MiniGPTLM(config)
input_ids = torch.tensor([[1, 2, 3, 4]])
logits, loss, attention = model(input_ids, labels=input_ids, return_attention=True)
attention_map = attention[0, 0].detach().cpu().numpy()
attention_map

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(attention_map, annot=True, cmap='Blues')
plt.title('Track B Attention Heatmap (Head 0)')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.tight_layout()
plt.show()